In [1]:
%cd /mlx_devbox/users/janne.spijkervet/repo/333/samantha/
%load_ext autoreload
%autoreload 2

/mlx_devbox/users/janne.spijkervet/repo/333/samantha


In [45]:
import os
from samantha.utils.hdfs_helper import hdfs_ls, get

# arnold_task_id = 558104
# hdfs_ckpt_dir = f"hdfs://harunava/home/byte_arnold_va/data/lab/audio/soundstorm/tasks/{arnold_task_id}/trials"
# checkpoints = list(
#     filter(lambda f: ".ckpt" in f and "step=" in f, hdfs_ls(f"-R {hdfs_ckpt_dir}"))
# )
# checkpoints.reverse()

# get(checkpoints[0], "model.ckpt")

get("hdfs://harunava/home/byte_arnold_va/data/lab/audio/soundstorm/tasks/558104/trials/2808839/output/soundstorm/baseline/checkpoints/epoch=0-step=21500.ckpt", "558104.ckpt")

True

In [46]:
from recipes.soundstorm.lightning.soundstorm import SoundStorm

device = "cuda"
soundstorm = SoundStorm.load_from_checkpoint("558104.ckpt").to(device)

/usr/local/lib/python3.9/dist-packages/pytorch_lightning/utilities/parsing.py:197: UserWarning: Attribute 'masking_scheme' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['masking_scheme'])`.
  rank_zero_warn(


In [27]:
import torch
import pandas as pd

semantic_tokens = torch.load("/mnt/bn/audio-diffusion/assets/semantic_decoder_gen_samples_100x250.pt")
semantic_tokens = semantic_tokens[:, None].to(device)


df = pd.read_csv("/mnt/bn/audio-diffusion/data/google_prompts/text_prompt_collection_20230523.csv")
text_prompts = df["text"].tolist()[:len(semantic_tokens)]

In [47]:
import re
import unicodedata

def slugify(value, allow_unicode=False):
    """
    Taken from https://github.com/django/django/blob/master/django/utils/text.py
    Convert to ASCII if 'allow_unicode' is False. Convert spaces or repeated
    dashes to single dashes. Remove characters that aren't alphanumerics,
    underscores, or hyphens. Convert to lowercase. Also strip leading and
    trailing whitespace, dashes, and underscores.
    """
    value = str(value)
    if allow_unicode:
        value = unicodedata.normalize("NFKC", value)
    else:
        value = (
            unicodedata.normalize("NFKD", value)
            .encode("ascii", "ignore")
            .decode("ascii")
        )
    value = re.sub(r"[^\w\s-]", "", value.lower())
    return re.sub(r"[-\s]+", "-", value).strip("-_")



In [48]:
import torchaudio
import os
from recipes.soundstorm.inference.semantic2audio import generate

# iterations = [32,32,32,32,8,8,8,8,8,8,8,8]
# score_strategies = ["random", "random", "random", "random", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit"]

iterations = [32,32,32,32,1,1,1,1,1,1,1,1]
score_strategies = ["random", "random", "random", "random", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit", "maskgit"]

out_dir = "generated_2808839"
os.makedirs(out_dir, exist_ok=True)
            
for st, text in zip(semantic_tokens, text_prompts):
    audio = generate(soundstorm, st, max_seq_len=500, iterations=iterations, score_strategies=score_strategies)    
    torchaudio.save(os.path.join(out_dir, f"{slugify(text)}.wav"), audio[0].cpu(), 24000)

Iteratively decoding audio tokens...: 100%|█████| 12/12 [00:09<00:00,  1.20it/s]
